In [1]:
import json 

with open("aiod_model.json", "r") as f:
    conceptual_model = json.load(f)

In [98]:
for i in range(len(conceptual_model['classes'])):
    if conceptual_model['classes'][i]['name'] == 'Event':
        print(i)

54


In [129]:
conceptual_model['classes'][54].keys()

dict_keys(['name', 'equivalent_classes', 'direct_properties', 'inherited_properties'])

Generate schema

In [12]:
# Jupyter only looks in its default paths (sys.path does not automatically include sibling directories).
import sys
import os

# Define the absolute path to `aiod/src`
SRC_PATH = os.path.abspath("../src")

# Add it to Python's module search path if it's not already there
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(f"✅ Added {SRC_PATH} to sys.path")

✅ Added /home/taniya_das/Documents/AIOD-rest-api/src to sys.path


In [48]:
# Example schema generation
from database.model.ai_asset.ai_asset import AIAsset

json.loads(AIAsset.schema_json())


{'title': 'AIAsset',
 'type': 'object',
 'properties': {'platform': {'title': 'Platform',
   'description': 'The external platform from which this resource originates. Leave empty if this item originates from AIoD. If platform is not None, the platform_resource_identifier should be set as well.',
   'maxLength': 64,
   'example': 'example',
   'type': 'string'},
  'platform_resource_identifier': {'title': 'Platform Resource Identifier',
   'description': "A unique identifier issued by the external platform that's specified in 'platform'. Leave empty if this item is not part of an external platform. For example, for HuggingFace, this should be the <namespace>/<dataset_name>, and for Openml, the OpenML identifier.",
   'maxLength': 256,
   'example': '1',
   'type': 'string'},
  'identifier': {'title': 'Identifier', 'type': 'integer'},
  'date_deleted': {'title': 'Date Deleted',
   'type': 'string',
   'format': 'date-time'},
  'aiod_entry_identifier': {'title': 'Aiod Entry Identifier',


In [ ]:
# generate complete schema 

import os
import json
import importlib.util
from pathlib import Path
from pydantic import BaseModel
from sqlmodel import SQLModel

# base directory 
BASE_DIR = "database/model"
OUTPUT_FILE = "schemas.json"

# dictionary to hold all schemas
all_schemas = {}
read_error = {}

for root, _, files in os.walk(os.path.join(SRC_PATH, BASE_DIR)):
    for file in files:
        if file.endswith(".py") and not file.startswith("__"):
            
            module_path = os.path.join(root, file)
            module_name = Path(module_path).with_suffix("").as_posix().split("/")[-1]
            # print(module_name)
            
            try:
               
                spec = importlib.util.spec_from_file_location(module_name, module_path)
                module = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(module)

                
                for attr_name in dir(module):
                    
                    SQLModel.metadata.clear()  
                    
                    attr = getattr(module, attr_name)
                    # Only process Pydantic models (subclasses of BaseModel)
                    if isinstance(attr, type) and issubclass(attr, BaseModel) and attr is not BaseModel:
                        
                        if attr.__name__.endswith("Base") or attr.__name__.endswith("Table") or attr.__name__.endswith("ORM"):  # Exclude abstract Base classes
                            continue
                     
                        schema = attr.schema_json()
                        all_schemas[attr.__name__] = json.loads(schema)

            except Exception as e:
                print(f"Error processing {module_name}: {e}")
                read_error.update({module_name:e})


# with open(OUTPUT_FILE, "w") as f:
#     json.dump(all_schemas, f, indent=4)

# print(f"All schemas have been written to {OUTPUT_FILE}")


In [86]:
read_error

{'location': TypeError('issubclass() arg 1 must be a class')}

In [126]:
all_schemas["Event"].keys()

dict_keys(['title', 'type', 'properties', 'required'])

In [141]:

generated_schema = all_schemas.copy()

def preprocess_generated_schema(generated_schema):
    """
    Preprocess the generated schema to:
    - Rename "title" to "name"
    - Add an empty "equivalent_classes" field
    - Convert "properties" into a list of dictionaries, renaming "title" to "name"
    """
    preprocessed_schema = []

    for entity_name, schema in generated_schema.items():
        if "properties" not in schema:
            continue  # Skip if no properties

        processed_properties = []

        for prop_name, prop_details in schema["properties"].items():
            # Convert property title to "name" and keep the rest of the metadata
            processed_property = prop_details.copy()
            processed_property["name"] = processed_property.pop("title", prop_name)

            processed_properties.append(processed_property)

        # Store the processed entity with new structure
        preprocessed_schema.append({
            "name": schema["title"],  # Rename "title" to "name"
            "equivalent_classes": [],  # Add empty equivalent_classes
            "properties": processed_properties,  # Use list of dicts for properties
            "required": schema.get("required", [])
        })

    return preprocessed_schema


# Apply preprocessing
preprocessed_generated_schema = preprocess_generated_schema(generated_schema)

